# Lexicase Selection: Simple Evolutionary Algorithm Demo

This notebook demonstrates how to use lexicase selection in a complete evolutionary algorithm for a simple symbolic regression task.

**Goal**: Evolve a polynomial that approximates a target function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lexicase import lexicase_selection, epsilon_lexicase_selection

np.random.seed(42)

## 1. Define the Problem

We'll evolve polynomials to approximate the function: `f(x) = x^3 - 2x^2 + x - 3`

Each individual is a vector of polynomial coefficients `[a0, a1, a2, a3]` representing:
`a0 + a1*x + a2*x^2 + a3*x^3`

In [ ]:
# Target function: x^3 - 2x^2 + x - 3
def target_function(x):
    return x**3 - 2*x**2 + x - 3

# Generate test cases (input points)
n_test_cases = 20
test_inputs = np.linspace(-2, 2, n_test_cases)
test_outputs = target_function(test_inputs)

# Visualize the target function
x_dense = np.linspace(-2, 2, 100)
plt.figure(figsize=(10, 4))
plt.plot(x_dense, target_function(x_dense), 'b-', linewidth=2, label='Target: $x^3 - 2x^2 + x - 3$')
plt.scatter(test_inputs, test_outputs, c='red', s=50, zorder=5, label='Test cases')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Target Function and Test Cases')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Number of test cases: {n_test_cases}")
print(f"Target coefficients: [-3, 1, -2, 1] (constant, x, x^2, x^3)")

## 2. Define Individual Representation and Evaluation

In [ ]:
def evaluate_polynomial(coefficients, x):
    """Evaluate polynomial with given coefficients at point x.
    
    coefficients: [a0, a1, a2, a3] for a0 + a1*x + a2*x^2 + a3*x^3
    """
    result = 0
    for i, coef in enumerate(coefficients):
        result += coef * (x ** i)
    return result


def compute_fitness_matrix(population, test_inputs, test_outputs):
    """Compute fitness matrix for entire population.
    
    Returns:
        Matrix of shape (n_individuals, n_test_cases)
        Higher values = better (we use negative error)
    """
    n_individuals = len(population)
    n_cases = len(test_inputs)
    
    fitness_matrix = np.zeros((n_individuals, n_cases))
    
    for i, individual in enumerate(population):
        for j, (x, y_target) in enumerate(zip(test_inputs, test_outputs)):
            y_pred = evaluate_polynomial(individual, x)
            # Fitness = negative absolute error (higher is better)
            error = abs(y_pred - y_target)
            fitness_matrix[i, j] = -error
    
    return fitness_matrix


# Test with a random individual
test_individual = np.random.uniform(-5, 5, 4)
print(f"Test individual coefficients: {test_individual.round(2)}")

fitness = compute_fitness_matrix([test_individual], test_inputs, test_outputs)
print(f"Fitness on each test case: {fitness[0].round(2)}")
print(f"Mean fitness: {fitness[0].mean():.2f}")

## 3. Define Genetic Operators

In [ ]:
def create_random_individual(n_coefficients=4):
    """Create a random polynomial individual."""
    return np.random.uniform(-5, 5, n_coefficients)


def mutate(individual, mutation_rate=0.3, mutation_strength=0.5):
    """Mutate an individual by adding Gaussian noise to coefficients."""
    child = individual.copy()
    for i in range(len(child)):
        if np.random.random() < mutation_rate:
            child[i] += np.random.normal(0, mutation_strength)
    return child


def crossover(parent1, parent2):
    """Single-point crossover between two parents."""
    point = np.random.randint(1, len(parent1))
    child = np.concatenate([parent1[:point], parent2[point:]])
    return child


# Test operators
parent1 = create_random_individual()
parent2 = create_random_individual()
child = crossover(parent1, parent2)
mutated = mutate(child)

print(f"Parent 1:  {parent1.round(2)}")
print(f"Parent 2:  {parent2.round(2)}")
print(f"Child:     {child.round(2)}")
print(f"Mutated:   {mutated.round(2)}")

## 4. Run the Evolutionary Algorithm

In [ ]:
def run_evolution(population_size=100, generations=100, selection_method='lexicase'):
    """
    Run evolutionary algorithm with specified selection method.
    
    Args:
        population_size: Number of individuals in population
        generations: Number of generations to evolve
        selection_method: 'lexicase', 'epsilon_lexicase', or 'tournament'
    
    Returns:
        best_individual, best_fitness_history, diversity_history
    """
    # Initialize population
    population = [create_random_individual() for _ in range(population_size)]
    
    best_fitness_history = []
    mean_fitness_history = []
    diversity_history = []
    best_individual_history = []
    
    for gen in range(generations):
        # Evaluate population
        fitness_matrix = compute_fitness_matrix(population, test_inputs, test_outputs)
        
        # Track statistics
        mean_fitness = np.mean(fitness_matrix)
        individual_mean_fitness = np.mean(fitness_matrix, axis=1)
        best_idx = np.argmax(individual_mean_fitness)
        best_fitness = individual_mean_fitness[best_idx]
        
        best_fitness_history.append(best_fitness)
        mean_fitness_history.append(mean_fitness)
        best_individual_history.append(population[best_idx].copy())
        
        # Measure diversity (standard deviation of coefficients)
        pop_array = np.array(population)
        diversity = np.mean(np.std(pop_array, axis=0))
        diversity_history.append(diversity)
        
        # Selection
        if selection_method == 'lexicase':
            selected_indices = lexicase_selection(fitness_matrix, population_size, seed=gen)
        elif selection_method == 'epsilon_lexicase':
            selected_indices = epsilon_lexicase_selection(fitness_matrix, population_size, seed=gen)
        elif selection_method == 'tournament':
            # Simple tournament selection for comparison
            selected_indices = []
            for _ in range(population_size):
                tournament = np.random.choice(population_size, size=3, replace=False)
                tournament_fitness = individual_mean_fitness[tournament]
                winner = tournament[np.argmax(tournament_fitness)]
                selected_indices.append(winner)
            selected_indices = np.array(selected_indices)
        
        # Create next generation
        new_population = []
        for i in range(0, population_size, 2):
            parent1 = population[selected_indices[i]]
            parent2 = population[selected_indices[min(i+1, population_size-1)]]
            
            # Crossover
            if np.random.random() < 0.7:  # 70% crossover rate
                child1 = crossover(parent1, parent2)
                child2 = crossover(parent2, parent1)
            else:
                child1 = parent1.copy()
                child2 = parent2.copy()
            
            # Mutation
            child1 = mutate(child1)
            child2 = mutate(child2)
            
            new_population.extend([child1, child2])
        
        population = new_population[:population_size]
        
        # Print progress
        if gen % 20 == 0 or gen == generations - 1:
            print(f"Gen {gen:3d}: Best fitness = {best_fitness:.3f}, Diversity = {diversity:.3f}")
    
    return best_individual_history[-1], best_fitness_history, mean_fitness_history, diversity_history

In [ ]:
# Run evolution with lexicase selection
print("Running with LEXICASE selection...")
print("=" * 50)
np.random.seed(42)
best_lex, fitness_lex, mean_lex, diversity_lex = run_evolution(
    population_size=100, 
    generations=80, 
    selection_method='lexicase'
)

print("\n" + "=" * 50)
print("Running with EPSILON LEXICASE selection...")
print("=" * 50)
np.random.seed(42)
best_eps, fitness_eps, mean_eps, diversity_eps = run_evolution(
    population_size=100, 
    generations=80, 
    selection_method='epsilon_lexicase'
)

print("\n" + "=" * 50)
print("Running with TOURNAMENT selection...")
print("=" * 50)
np.random.seed(42)
best_tour, fitness_tour, mean_tour, diversity_tour = run_evolution(
    population_size=100, 
    generations=80, 
    selection_method='tournament'
)

## 5. Compare Results

In [ ]:
# Plot evolution progress
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Best fitness over time
ax1 = axes[0, 0]
ax1.plot(fitness_lex, label='Lexicase', linewidth=2)
ax1.plot(fitness_eps, label='Epsilon Lexicase', linewidth=2)
ax1.plot(fitness_tour, label='Tournament', linewidth=2)
ax1.set_xlabel('Generation')
ax1.set_ylabel('Best Fitness (negative error)')
ax1.set_title('Best Fitness Over Time (higher = better)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Diversity over time
ax2 = axes[0, 1]
ax2.plot(diversity_lex, label='Lexicase', linewidth=2)
ax2.plot(diversity_eps, label='Epsilon Lexicase', linewidth=2)
ax2.plot(diversity_tour, label='Tournament', linewidth=2)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Population Diversity')
ax2.set_title('Population Diversity Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Best solutions found
ax3 = axes[1, 0]
x_plot = np.linspace(-2, 2, 100)
ax3.plot(x_plot, target_function(x_plot), 'k-', linewidth=3, label='Target', alpha=0.7)

y_lex = np.array([evaluate_polynomial(best_lex, x) for x in x_plot])
y_eps = np.array([evaluate_polynomial(best_eps, x) for x in x_plot])
y_tour = np.array([evaluate_polynomial(best_tour, x) for x in x_plot])

ax3.plot(x_plot, y_lex, '--', linewidth=2, label='Lexicase')
ax3.plot(x_plot, y_eps, '--', linewidth=2, label='Epsilon Lexicase')
ax3.plot(x_plot, y_tour, '--', linewidth=2, label='Tournament')

ax3.scatter(test_inputs, test_outputs, c='red', s=30, zorder=5, alpha=0.5)
ax3.set_xlabel('x')
ax3.set_ylabel('f(x)')
ax3.set_title('Best Solutions Found')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Coefficient comparison
ax4 = axes[1, 1]
target_coeffs = [-3, 1, -2, 1]
x_pos = np.arange(4)
width = 0.2

ax4.bar(x_pos - 1.5*width, target_coeffs, width, label='Target', alpha=0.8)
ax4.bar(x_pos - 0.5*width, best_lex, width, label='Lexicase', alpha=0.8)
ax4.bar(x_pos + 0.5*width, best_eps, width, label='Epsilon Lex', alpha=0.8)
ax4.bar(x_pos + 1.5*width, best_tour, width, label='Tournament', alpha=0.8)

ax4.set_xlabel('Coefficient')
ax4.set_ylabel('Value')
ax4.set_title('Evolved vs Target Coefficients')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(['$a_0$', '$a_1$', '$a_2$', '$a_3$'])
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Print final results
print("FINAL RESULTS")
print("=" * 60)
print(f"Target coefficients: {[-3, 1, -2, 1]}")
print()

methods = [
    ('Lexicase', best_lex, fitness_lex[-1]),
    ('Epsilon Lexicase', best_eps, fitness_eps[-1]),
    ('Tournament', best_tour, fitness_tour[-1]),
]

for name, coeffs, fitness in methods:
    # Compute error from target
    target = np.array([-3, 1, -2, 1])
    coeff_error = np.sqrt(np.mean((coeffs - target)**2))
    print(f"{name:20s}: {coeffs.round(3)}")
    print(f"{'':20s}  Final fitness: {fitness:.4f}")
    print(f"{'':20s}  Coefficient RMSE: {coeff_error:.4f}")
    print()

## 6. Understanding Lexicase Selection Behavior

Let's visualize how lexicase selection maintains diversity by looking at specialist selection.

In [ ]:
# Create a population with specialists
np.random.seed(42)

# Create some "specialist" individuals that are good at specific regions
specialists = [
    # Good at negative x (left region)
    np.array([-3.5, 1.2, -1.8, 0.9]),
    np.array([-2.8, 0.8, -2.1, 1.1]),
    # Good at positive x (right region)
    np.array([-3.2, 1.1, -2.2, 1.0]),
    np.array([-2.9, 0.9, -1.9, 0.95]),
    # Generalist (mediocre everywhere)
    np.array([-2.5, 0.5, -1.5, 0.8]),
    np.array([-3.0, 1.0, -2.0, 1.0]),  # Near-perfect
]

# Add some random individuals
for _ in range(10):
    specialists.append(create_random_individual())

# Compute fitness
fitness_matrix = compute_fitness_matrix(specialists, test_inputs, test_outputs)

# Visualize fitness profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of fitness by case
ax1 = axes[0]
im = ax1.imshow(fitness_matrix[:6], aspect='auto', cmap='RdYlGn')
ax1.set_xlabel('Test Case (x value)')
ax1.set_ylabel('Individual')
ax1.set_title('Fitness by Test Case (Green = Good)')
ax1.set_yticks(range(6))
ax1.set_yticklabels(['Specialist 1', 'Specialist 2', 'Specialist 3', 
                     'Specialist 4', 'Generalist', 'Near-Perfect'])
plt.colorbar(im, ax=ax1, label='Fitness')

# Selection frequency comparison
ax2 = axes[1]

# Run many selections
n_trials = 500
lex_counts = np.zeros(len(specialists))
tour_counts = np.zeros(len(specialists))

for trial in range(n_trials):
    # Lexicase
    lex_sel = lexicase_selection(fitness_matrix, 4, seed=trial)
    for idx in lex_sel:
        lex_counts[idx] += 1
    
    # Tournament
    mean_fitness = np.mean(fitness_matrix, axis=1)
    for _ in range(4):
        tournament = np.random.choice(len(specialists), size=3, replace=False)
        winner = tournament[np.argmax(mean_fitness[tournament])]
        tour_counts[winner] += 1

lex_freq = lex_counts / (n_trials * 4)
tour_freq = tour_counts / (n_trials * 4)

x_pos = np.arange(6)
width = 0.35
ax2.bar(x_pos - width/2, lex_freq[:6], width, label='Lexicase', alpha=0.8)
ax2.bar(x_pos + width/2, tour_freq[:6], width, label='Tournament', alpha=0.8)
ax2.set_xlabel('Individual')
ax2.set_ylabel('Selection Frequency')
ax2.set_title('Selection Frequency Comparison')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['Spec 1', 'Spec 2', 'Spec 3', 'Spec 4', 'General', 'Perfect'])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Observation:")
print("- Tournament selection heavily favors the near-perfect individual")
print("- Lexicase selection maintains diversity by giving specialists a chance")
print("- This helps explore the search space more thoroughly")

## Summary

This notebook demonstrated:

1. **Problem Setup**: Symbolic regression to evolve polynomials
2. **Fitness Matrix**: How to evaluate individuals on multiple test cases
3. **Selection Methods**: Comparing lexicase, epsilon lexicase, and tournament
4. **Evolution**: Running a complete evolutionary algorithm
5. **Diversity Maintenance**: How lexicase preserves specialists

### Key Takeaways

- **Lexicase selection** evaluates individuals case-by-case, maintaining diversity
- **Epsilon lexicase** is more tolerant, allowing "good enough" individuals to pass
- **Tournament selection** tends to converge quickly but may lose diversity
- The fitness matrix convention: `(n_individuals, n_test_cases)` with higher = better